# Análisis de sentimientos en reseñas de películas

En este proyecto entreno y comparo distintos modelos de clasificación para determinar si una reseña de IMDB expresa un sentimiento **positivo** o **negativo**. El flujo incluye exploración, limpieza de texto, vectorización TF-IDF, entrenamiento y evaluación con accuracy, precision, recall, F1 y matrices de confusión.

In [ ]:
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report,
                             confusion_matrix)

warnings.filterwarnings('ignore', category=MarkupResemblesLocatorWarning)
sns.set_theme(style='darkgrid')
RANDOM_STATE = 42

## 1. Carga y exploración de los datos

In [ ]:
df = pd.read_csv('./data/IMDB Dataset.csv')
print(f'Dimensiones del conjunto original: {df.shape}')
display(df.head())
df.info()

In [ ]:
print('Valores nulos:')
display(df.isna().sum())
print(f"Reseñas duplicadas: {df.duplicated(subset='review').sum()}")
display(df['sentiment'].value_counts())

sns.countplot(data=df, x='sentiment', hue='sentiment', legend=False, palette='Set2')
plt.title('Distribución de sentimientos')
plt.xlabel('Sentimiento')
plt.ylabel('Número de reseñas')
plt.show()

El conjunto original contiene 50,000 reseñas balanceadas: 25,000 positivas y 25,000 negativas. Para reducir el tiempo de entrenamiento utilizo una muestra equilibrada de 10,000 reseñas, como indica la actividad.

In [ ]:
df_pos = df[df['sentiment'] == 'positive'].sample(5000, random_state=RANDOM_STATE)
df_neg = df[df['sentiment'] == 'negative'].sample(5000, random_state=RANDOM_STATE)
df_reviews = (
    pd.concat([df_pos, df_neg], ignore_index=True)
      .sample(frac=1, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)
df_reviews['sentiment'].value_counts()

## 2. Limpieza y preparación del texto

In [ ]:
def limpiar_texto(texto):
    texto = BeautifulSoup(str(texto), 'html.parser').get_text(' ')
    texto = texto.lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df_reviews['review_clean'] = df_reviews['review'].apply(limpiar_texto)
df_reviews[['review', 'review_clean', 'sentiment']].head()

In [ ]:
train_x, test_x, train_y, test_y = train_test_split(
    df_reviews['review_clean'],
    df_reviews['sentiment'],
    test_size=0.33,
    random_state=RANDOM_STATE,
    stratify=df_reviews['sentiment']
)

print(f'Entrenamiento: {train_x.shape[0]} reseñas')
print(f'Prueba: {test_x.shape[0]} reseñas')
display(train_y.value_counts())

## 3. Vectorización TF-IDF

TF-IDF asigna mayor peso a términos frecuentes dentro de una reseña pero poco comunes en el resto del corpus. Se eliminan palabras vacías en inglés y se incluyen unigramas y bigramas. El vectorizador se ajusta solamente con los datos de entrenamiento para evitar fuga de información.

In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

print(f'Matriz de entrenamiento: {train_x_vector.shape}')
print(f'Matriz de prueba: {test_x_vector.shape}')
print(f'Tipo: {type(train_x_vector)}')

In [ ]:
# Palabras más relevantes en una reseña de ejemplo
ejemplo_tfidf = pd.Series(
    train_x_vector[0].toarray().ravel(),
    index=tfidf.get_feature_names_out()
).sort_values(ascending=False)
ejemplo_tfidf.head(15)

## 4. Funciones de evaluación

In [ ]:
resultados = []

def evaluar_modelo(nombre, y_real, y_predicho, tiempo):
    metricas = {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_real, y_predicho),
        'Precision': precision_score(y_real, y_predicho, pos_label='positive'),
        'Recall': recall_score(y_real, y_predicho, pos_label='positive'),
        'F1': f1_score(y_real, y_predicho, pos_label='positive'),
        'Tiempo (s)': tiempo
    }
    resultados.append(metricas)
    print(f'\n{nombre}')
    print('-' * len(nombre))
    print(classification_report(y_real, y_predicho, digits=4))
    matriz = confusion_matrix(y_real, y_predicho, labels=['positive', 'negative'])
    sns.heatmap(
        matriz, annot=True, fmt='d', cmap='Blues',
        xticklabels=['positive', 'negative'],
        yticklabels=['positive', 'negative']
    )
    plt.title(f'Matriz de confusión: {nombre}')
    plt.xlabel('Predicción')
    plt.ylabel('Valor real')
    plt.show()
    return metricas

## 5. Modelo base: Máquina de Vectores de Soporte (SVM)

In [ ]:
inicio = time.perf_counter()
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)
pred_svc = svc.predict(test_x_vector)
tiempo_svc = time.perf_counter() - inicio
evaluar_modelo('SVM lineal', test_y, pred_svc, tiempo_svc)

## 6. Modelo alternativo: Regresión Logística

In [ ]:
inicio = time.perf_counter()
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(train_x_vector, train_y)
pred_log = log_reg.predict(test_x_vector)
tiempo_log = time.perf_counter() - inicio
evaluar_modelo('Regresión Logística', test_y, pred_log, tiempo_log)

## 7. Modelo alternativo: Árbol de Decisión

In [ ]:
inicio = time.perf_counter()
arbol = DecisionTreeClassifier(
    max_depth=30, min_samples_leaf=5, random_state=RANDOM_STATE
)
arbol.fit(train_x_vector, train_y)
pred_arbol = arbol.predict(test_x_vector)
tiempo_arbol = time.perf_counter() - inicio
evaluar_modelo('Árbol de Decisión', test_y, pred_arbol, tiempo_arbol)

## 8. Modelo alternativo: Gaussian Naive Bayes

`GaussianNB` necesita datos densos; convertir directamente toda la matriz TF-IDF consumiría demasiada memoria. Por eso primero aplico `TruncatedSVD`, que reduce la dimensionalidad conservando la estructura principal del texto.

In [ ]:
svd = TruncatedSVD(n_components=300, random_state=RANDOM_STATE)
train_x_reducido = svd.fit_transform(train_x_vector)
test_x_reducido = svd.transform(test_x_vector)
print(f'Varianza explicada acumulada: {svd.explained_variance_ratio_.sum():.2%}')

inicio = time.perf_counter()
gaussian_nb = GaussianNB()
gaussian_nb.fit(train_x_reducido, train_y)
pred_gnb = gaussian_nb.predict(test_x_reducido)
tiempo_gnb = time.perf_counter() - inicio
evaluar_modelo('Gaussian Naive Bayes', test_y, pred_gnb, tiempo_gnb)

## 9. Comparación de modelos

In [ ]:
comparacion = (
    pd.DataFrame(resultados)
      .sort_values(['F1', 'Accuracy'], ascending=False)
      .reset_index(drop=True)
)
display(comparacion.style.format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
    'Recall': '{:.4f}', 'F1': '{:.4f}', 'Tiempo (s)': '{:.2f}'
}))

comparacion_melt = comparacion.melt(
    id_vars='Modelo', value_vars=['Accuracy', 'Precision', 'Recall', 'F1'],
    var_name='Métrica', value_name='Puntuación'
)
plt.figure(figsize=(12, 6))
sns.barplot(data=comparacion_melt, x='Modelo', y='Puntuación', hue='Métrica')
plt.ylim(0, 1)
plt.title('Comparación del rendimiento de los modelos')
plt.xticks(rotation=15)
plt.show()

mejor_modelo = comparacion.iloc[0]
print(f"Mejor modelo por F1: {mejor_modelo['Modelo']} ({mejor_modelo['F1']:.4f})")

### Interpretación

La comparación se basa principalmente en **F1**, porque equilibra precisión y recall. También reviso accuracy, el desempeño por clase y los tiempos de entrenamiento. Los modelos lineales suelen ser adecuados para texto TF-IDF de alta dimensionalidad; el árbol puede sobreajustarse y GaussianNB requiere reducción dimensional. El código selecciona automáticamente el mejor modelo a partir de los resultados obtenidos, sin asumir la respuesta antes de ejecutar el experimento.

## 10. Prueba con reseñas nuevas

In [ ]:
modelos_sparse = {
    'SVM lineal': svc,
    'Regresión Logística': log_reg,
    'Árbol de Decisión': arbol
}

nombre_ganador = comparacion.iloc[0]['Modelo']
resenias_nuevas = [
    'I loved this movie, the acting was excellent and the story was moving.',
    'This was boring, predictable and a complete waste of time.',
    'The movie had good visuals but the story was disappointing.'
]
resenias_limpias = [limpiar_texto(texto) for texto in resenias_nuevas]
vectores_nuevos = tfidf.transform(resenias_limpias)

if nombre_ganador == 'Gaussian Naive Bayes':
    predicciones_nuevas = gaussian_nb.predict(svd.transform(vectores_nuevos))
else:
    predicciones_nuevas = modelos_sparse[nombre_ganador].predict(vectores_nuevos)

pd.DataFrame({
    'Reseña': resenias_nuevas,
    'Sentimiento predicho': predicciones_nuevas
})

## Conclusión

El procesamiento de lenguaje natural permite transformar texto no estructurado en variables numéricas mediante TF-IDF. Después de comparar SVM, Regresión Logística, Árbol de Decisión y Gaussian Naive Bayes, la elección final se realiza con base en F1, accuracy, equilibrio entre clases y costo de entrenamiento. Este procedimiento es más confiable que elegir un algoritmo solamente por intuición y permite aplicar el modelo ganador a reseñas que nunca vio durante el entrenamiento.